# Smart Home Location Selector
## Data Exploration & Recommendation System Demo

This notebook explores the smart home location dataset and demonstrates the recommendation engine.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✓ All libraries imported successfully')

## 2. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('smart_home_location_dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Dataset info
print('Dataset Info:')
print(df.info())
print(f'\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Cities and areas overview
print('Cities in dataset:')
cities_count = df['City'].value_counts()
print(cities_count)
print(f'\nTotal unique cities: {df["City"].nunique()}')
print(f'Total unique areas: {df["Area"].nunique()}')

## 3. Statistical Summary

In [ ]:
# Statistical summary of key metrics
summary_cols = ['Avg_Price_per_sqft', 'Avg_Rent', 'Safety_Score', 'Pollution_Index', 
                'Hospital_Distance_km', 'School_Distance_km', 'Metro_Distance_km']

print('Statistical Summary of Key Metrics:')
df[summary_cols].describe().round(2)

## 4. Price Distribution by City

In [ ]:
# Price distribution by city
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot for purchase price
df.boxplot(column='Avg_Price_per_sqft', by='City', ax=axes[0])
axes[0].set_title('Price per Sqft Distribution by City')
axes[0].set_ylabel('Price per Sqft (₹)')
axes[0].set_xlabel('City')

# Box plot for rent
df.boxplot(column='Avg_Rent', by='City', ax=axes[1])
axes[1].set_title('Monthly Rent Distribution by City')
axes[1].set_ylabel('Rent (₹)')
axes[1].set_xlabel('City')

plt.tight_layout()
plt.show()

print('Price statistics by city:')
df.groupby('City')[['Avg_Price_per_sqft', 'Avg_Rent']].mean().round(0)

## 5. Safety and Pollution Analysis

In [ ]:
# Safety vs Pollution scatter plot
fig, ax = plt.subplots(figsize=(12, 6))

cities = df['City'].unique()
colors = plt.cm.Set3(np.linspace(0, 1, len(cities)))

for i, city in enumerate(cities):
    city_data = df[df['City'] == city]
    ax.scatter(city_data['Safety_Score'], city_data['Pollution_Index'], 
              label=city, alpha=0.6, s=100, color=colors[i])

ax.set_xlabel('Safety Score (Higher = Safer)', fontsize=11)
ax.set_ylabel('Pollution Index (Lower = Better)', fontsize=11)
ax.set_title('Safety vs Pollution Index by City', fontsize=13, fontweight='bold')
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Safety & Pollution statistics by city:')
df.groupby('City')[['Safety_Score', 'Pollution_Index']].mean().round(2)

## 6. Distance Analysis (Proximity to Services)

In [ ]:
# Average distances by city
distance_cols = ['Hospital_Distance_km', 'School_Distance_km', 'Metro_Distance_km', 'Grocery_Distance_km']
distance_by_city = df.groupby('City')[distance_cols].mean()

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
distance_by_city.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Average Distance to Services by City', fontsize=13, fontweight='bold')
ax.set_ylabel('Distance (km)', fontsize=11)
ax.set_xlabel('City', fontsize=11)
ax.legend(title='Service Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Distance statistics by city:')
print(distance_by_city.round(2))

## 7. Internet Speed Distribution

In [ ]:
# Internet speed distribution
fig, ax = plt.subplots(figsize=(12, 6))

df.boxplot(column='Internet_Speed_Mbps', by='City', ax=ax)
ax.set_title('Internet Speed Distribution by City')
ax.set_ylabel('Speed (Mbps)')
ax.set_xlabel('City')
plt.tight_layout()
plt.show()

print('Internet speed statistics by city:')
df.groupby('City')['Internet_Speed_Mbps'].agg(['mean', 'min', 'max']).round(2)

## 8. Top 10 Best Locations Overall

In [ ]:
# Create a simple overall score
scaler = MinMaxScaler()

# Normalize metrics
df_scored = df.copy()

# Positive metrics (higher is better)
positive_metrics = ['Safety_Score', 'Internet_Speed_Mbps', 'Water_Availability_Score']
for col in positive_metrics:
    df_scored[f'{col}_norm'] = scaler.fit_transform(df_scored[[col]])

# Negative metrics (lower is better)
negative_metrics = ['Pollution_Index', 'Hospital_Distance_km', 'School_Distance_km', 'Metro_Distance_km', 'Avg_Price_per_sqft']
for col in negative_metrics:
    normalized = scaler.fit_transform(df_scored[[col]])
    df_scored[f'{col}_norm'] = 1 - normalized

# Calculate overall score
score_cols = [col for col in df_scored.columns if col.endswith('_norm')]
df_scored['Overall_Score'] = df_scored[score_cols].mean(axis=1)

# Top 10 locations
top_10 = df_scored.nlargest(10, 'Overall_Score')[['City', 'Area', 'Avg_Price_per_sqft', 'Safety_Score', 
                                                     'Pollution_Index', 'Metro_Distance_km', 'Overall_Score']]
top_10['Overall_Score'] = (top_10['Overall_Score'] * 100).round(2)
top_10.columns = ['City', 'Area', 'Price/sqft', 'Safety', 'Pollution', 'Metro(km)', 'Score(%)']

print('\nTop 10 Best Locations Overall:')
print(top_10.to_string(index=False))

## 9. Budget-Friendly Locations

In [ ]:
# Budget-friendly locations (low price, decent safety)
budget_friendly = df_scored[
    (df_scored['Avg_Price_per_sqft'] < df_scored['Avg_Price_per_sqft'].quantile(0.33)) &
    (df_scored['Safety_Score'] >= df_scored['Safety_Score'].quantile(0.5))